## Iterator

---

> **In one line.** An iterator $\mathcal{I}$ is a *stateful cursor* drawn from a collection $\mathcal{C}$ by a map $\text{iter} : \mathcal{C} \to \mathcal{I}$. It walks the data through a chain of cursor states $q_0, q_1, q_2, \dots$, at each step yielding one element $x \in X$ and advancing — all behind the uniform pair $\text{next}$ and $\text{has\_next}$, and without ever revealing how $\mathcal{C}$ is built inside.

### 1. The collection and its cursor

Start with a **collection** $\mathcal{C}$ — the data structure being traversed. It may be a list, a tree, a graph, a file, or a stream; the pattern does not care which. What it stores is the **element type** $X$: the individual items handed back one at a time, whether integers, strings, or node objects.

From $\mathcal{C}$ we manufacture an **iterator** $\mathcal{I}$, a small object whose entire job is to remember *where we are* and *how to move on*. This manufacturing step is itself a map,

$$\text{iter} : \mathcal{C} \longrightarrow \mathcal{I},$$

and the resulting $\mathcal{I}$ is best read as a tiny state machine. Let $Q$ be the set of all possible **cursor states** — the positions reachable within $\mathcal{C}$ — with a distinguished **initial state** $q_0 \in Q$ marking where traversal begins (typically the first element). A subset $F \subseteq Q$ collects the **final states**: once the cursor lands in $F$, every element has been seen and there is nothing left to yield.

### 2. The step: yield and advance

All the motion lives in a single **transition function** $\delta$. Given the current state $q_i$, it does two things at once — it produces the element sitting at that position, and it reports the next position the cursor should occupy:

$$\boxed{\,\delta(q_i) \;=\; (\,x_i,\; q_{i+1}\,)\,} \qquad \text{(yield element, advance cursor)}.$$

So the whole iterator is captured by the quadruple

$$\mathcal{I} \;=\; (\,Q,\; q_0,\; \delta,\; F\,).$$

Reading the data sequentially is just iterating $\delta$ from $q_0$ until a final state is reached. The element stream and the state progression unfold in lockstep:

$$\underbrace{q_0}_{\text{start}} \;\xrightarrow{\;\delta\;}\; (x_0,\, q_1) \;\xrightarrow{\;\delta\;}\; (x_1,\, q_2) \;\xrightarrow{\;\delta\;}\; \cdots \;\xrightarrow{\;\delta\;}\; (x_{k-1},\, q_k) \quad\text{with}\quad q_k \in F.$$

The two public operations map cleanly onto this machine. The **advance function** $\text{next}$ applies $\delta$ once — returning the current $x_i \in X$ and committing the move to $q_{i+1}$. The **termination check** $\text{has\_next}$ simply asks whether the current state has escaped $F$: it is `True` while elements remain and `False` the moment the traversal is complete.

$$\text{has\_next}(q_i) \;=\; \big[\, q_i \notin F \,\big].$$

### 3. Key conditions

1. **Separation** — $\mathcal{C}$ never exposes its internal structure. The caller is handed only $\mathcal{I}$, the cursor abstraction; the layout of the underlying container stays sealed away.
   $$\text{caller sees } \mathcal{I} \text{ only}, \qquad \text{internals of } \mathcal{C} \perp \text{caller}.$$
2. **Uniform interface** — the same `next()` and `has_next()` work over lists, trees, graphs, or streams alike. The shape of the collection is irrelevant to the caller, who programs against $\delta$ and $\text{has\_next}$ and nothing else.
3. **Termination** — when the cursor reaches a final state, $q \in F$, raising `StopIteration` signals the end. Python's `for` loop catches this automatically, so $F$ is exactly the boundary at which iteration halts.

&nbsp;

> 📖 A bookmark ($\mathcal{I}$) in a book ($\mathcal{C}$). The book can be organised however it likes. The bookmark just knows the current page ($q_i$) and how to move to the next ($\delta$). You never need to understand the binding to read sequentially.

### Exercise 07 — Custom Range Iterator

---

**Scenario:** Build a `CountDown` collection that counts from $n$ to 0. It must work with Python's `for` loop and `next()` by implementing the iterator protocol.

**Your task:** Implement `CountDown` with `__iter__` and `__next__` — the Python names for $\text{iter}$ and $\delta$.

```python
for n in CountDown(5):
    print(n)   # 5, 4, 3, 2, 1, 0 — δ(q_i) = (i, q_{i-1})
```

**Hints**

- `__iter__` returns `self` (the iterator is its own iterable). `__next__` implements $\delta$: return current value, decrement cursor, raise `StopIteration` when $q \in F$.
- `self._current` is exactly $q_i$ — the cursor state in the state machine.

In [ ]:
# --------------------------------
# Collection C is also its own iterator I — iter(C) returns self

class CountDown:
    def __init__(self, n):
        self._current = n                    # q_0: the initial cursor state

    def __iter__(self):                      # iter : C -> I
        # the iterator is its own iterable -> return self
        ...

    def __next__(self):                      # delta(q_i) = (x_i, q_{i+1})
        # if q in F (current < 0) -> raise StopIteration
        # else: capture x_i = self._current, advance cursor (q_{i-1}), return x_i
        ...

# --------------------------------
for n in CountDown(5):
    print(n)                                 # expect 5, 4, 3, 2, 1, 0

### Exercise 08 — Binary Tree In-Order Iterator

---

**Scenario:** A binary tree ($\mathcal{C}$) stores values. Traverse all values in sorted (in-order) sequence using `for` — without exposing the tree's internal node structure.

**Your task:** Build `BinaryTree` and an `InOrderIterator` traversing left-root-right. The tree's node class is never visible to the caller.

```python
tree = BinaryTree()
for v in [5, 3, 8, 1, 4]:
    tree.insert(v)
for v in tree:
    print(v)   # 1, 3, 4, 5, 8 — in-order, the node type stays hidden
```

**Hints**

- Use a stack to simulate recursive in-order traversal — the stack is $q_i$, the cursor state encoding "where I am in the traversal".
- The tree's `__iter__` returns `InOrderIterator(self.root)` — the internal node type is encapsulated inside $\mathcal{I}$, never exposed.

In [ ]:
# --------------------------------
# Internal node type — never visible to the caller (Separation)

class _Node:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

# --------------------------------
# The iterator I = (Q, q_0, delta, F) — q is the stack of pending nodes

class InOrderIterator:
    def __init__(self, root):
        self._stack = []                     # q: cursor state (the traversal stack)
        self._push_left(root)                # establishes q_0

    def _push_left(self, node):
        # push node, then all its left descendants onto the stack
        while node is not None:
            self._stack.append(node)
            node = node.left

    def __iter__(self):
        return self

    def __next__(self):                      # delta(q_i) = (x_i, q_{i+1})
        # if stack empty -> q in F -> raise StopIteration
        # else: pop node = x_i, then _push_left(node.right) to advance to q_{i+1}
        # return node.value
        ...

# --------------------------------
# The collection C — exposes only iter(C); the node type stays hidden

class BinaryTree:
    def __init__(self):
        self._root = None

    def insert(self, value):
        self._root = self._insert(self._root, value)

    def _insert(self, node, value):
        if node is None:
            return _Node(value)
        if value < node.value:
            node.left = self._insert(node.left, value)
        else:
            node.right = self._insert(node.right, value)
        return node

    def __iter__(self):                      # iter : C -> I (node type encapsulated)
        return InOrderIterator(self._root)

# --------------------------------
tree = BinaryTree()
for v in [5, 3, 8, 1, 4]:
    tree.insert(v)
print(list(tree))                            # expect [1, 3, 4, 5, 8]